In [1]:
## Import libraries
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support,classification_report
import os
from sklearn.cluster import AgglomerativeClustering
import numpy as np

In [2]:
## Store all the results in dictionary

# Define the data types for the columns
dtype_dict = {'instance_id': str, 'dataset': str, 'mention_type': str, 'label': int, 'prediction': float, 'subtopic': str}

# Get a list of all CSV files in the "results" folder
folder_path = "results"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
dict_predictions = {}
# Iterate through each CSV file and store full file in dictionary

for csv_file in csv_files:
    file_path = os.path.join(folder_path, csv_file)
    
    df = pd.read_csv(file_path, dtype=dtype_dict, low_memory=False).fillna("NA")


    dict_predictions[csv_file] = df.copy()



In [3]:
def compute_metrics_pi(df):

    df_filtered= df.query("dataset=='mrpc' or dataset=='how'").copy()
    df_filtered['prediction']=(df_filtered.prediction>0).astype(int)
    
    grouped_metrics = df_filtered.groupby(['dataset']).apply(lambda x: pd.Series(
        precision_recall_fscore_support(x['label'], x['prediction'], average='weighted'),
        index=['Precision', 'Recall', 'F1 Score', 'Support']
    ))
    return grouped_metrics


In [4]:
# #Find better output format
# for k,v in dict_predictions.items():

#     print(f"File: {k}")
#     print(compute_metrics_pi(v))
#     print("=" * 50)  # Separator between files

# Create clusters for NewsWCL50

In [5]:
def extract_mention_i(text,i):
    text_list = text.split("[SEP]")
    return text_list[i]

In [6]:
def predict_clusters(input_df):
    df_exp = input_df.copy()

    #Extract mentions id
    df_exp["mention1_id"] = df_exp.instance_id.apply(lambda x:extract_mention_i(x,1) )
    df_exp["mention2_id"] = df_exp.instance_id.apply(lambda x:extract_mention_i(x,2) )

    # 1. Turn logit into probability
    df_exp['predicted_prob'] = 1 / (1 + np.exp(-df_exp['prediction']))
    # 2. Compute distance
    df_exp['distance'] = 1-df_exp['predicted_prob'] 
    # 3. Map mentions to indices

    first_set = set(df_exp.mention1_id)
    second_set = set(df_exp.mention2_id)
    union_set = first_set.union(second_set)
    num_mentions = len(union_set)

    mention_index_dict = {value: idx for idx, value in enumerate(union_set)}
    index_mention_dict = {idx: value for idx, value in enumerate(union_set)}

    df_exp["mention1_id_index"] =  df_exp["mention1_id"].map(mention_index_dict)
    df_exp["mention2_id_index"] =  df_exp["mention2_id"].map(mention_index_dict)
    # Create square matrix of 0s (minimum distance) and replace combinations with actual distances
    distance_matrix = np.zeros((num_mentions,num_mentions))

    mention1_indices = df_exp["mention1_id_index"].values
    mention2_indices = df_exp["mention2_id_index"].values
    distances = df_exp["distance"].values

    distance_matrix[mention1_indices, mention2_indices] = distances
    distance_matrix[mention2_indices, mention1_indices] = distances

    #Name matrix with mention ids
    final_distance_matrix = pd.DataFrame(distance_matrix)

    column_names = []
    for col in final_distance_matrix.columns:
        column_names.append(index_mention_dict[col])

    final_distance_matrix.columns=column_names

    #Apply agglomerative clustering
    clustering = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average',
                                            distance_threshold=0.7)
    fit = clustering.fit(final_distance_matrix)
    return pd.DataFrame({'mention_id':fit.feature_names_in_,"predicted_cluster":fit.labels_})

In [7]:
dict_predicted_clusters = {}
for file_name, pairwise_predictions in dict_predictions.items():
    list_predicted_clusters_per_file = []
    #Choose one df
    chosen_df = pairwise_predictions.query("dataset=='NewsWCL50-prep'")
    unique_mention_types = chosen_df.mention_type.unique()
    dict_combine_topics = {}
    #new file key
    new_file_key = file_name.split(".")[0].replace('predicted_pairs','cluster')

    #Create all the pairs mention type, subtopic
    for mention_type in unique_mention_types:
        dict_combine_topics[mention_type] = chosen_df[chosen_df['mention_type']==mention_type].subtopic.unique()

    for entity_type,subtopics in dict_combine_topics.items():
        for subtopic in subtopics:
            chosen_df_subtopic = chosen_df[(chosen_df['mention_type'] == entity_type) & (chosen_df['subtopic'] == subtopic)]
            predicted_clusters_df = predict_clusters(chosen_df_subtopic)
            predicted_clusters_df = predicted_clusters_df.rename(columns={'predicted_cluster':new_file_key})
            list_predicted_clusters_per_file.append(predicted_clusters_df)
    
    dict_predicted_clusters[new_file_key] = pd.concat(list_predicted_clusters_per_file)


In [10]:
merged_df = list(dict_predicted_clusters.values())[0]
for key, df in list(dict_predicted_clusters.items())[1:]:
    merged_df = merged_df.merge(df, on='mention_id', how='outer')

In [12]:
merged_df

,mention_id,cluster_exp1_seq_sampling,cluster_exp1_sim_sampling,cluster_exp2_seq_sampling,cluster_exp2_sim_sampling,cluster_exp3_seq_sampling,cluster_exp3_sim_sampling,cluster_exp4_seq_sampling,cluster_exp4_sim_sampling
0,0_M_9_22_oNV7,46,7,0,0,63,40,77,59
1,0_L_14_28_cQ2y,1,12,0,0,9,12,1,2
2,0_L_42_9_nsBw,23,54,0,0,84,3,6,10
3,0_M_14_3_hRyv,2,12,0,0,20,5,1,2
4,0_L_18_6_mCHg,24,5,0,0,34,12,59,29
...,...,...,...,...,...,...,...,...,...
5114,2_L_31_36_Qx6t,12,26,0,0,66,37,72,35
5115,2_RR_11_11_RzG7,3,6,0,0,0,86,13,4
5116,2_L_6_4_QpLW,4,14,0,0,23,3,1,50
5117,2_L_4_4_myML,10,8,0,0,16,0,1,53


In [14]:
column_to_check = 'cluster_exp1_seq_sampling'
column_to_compare = 'cluster_exp4_sim_sampling'
are_values_equal = (merged_df[column_to_check] == merged_df[column_to_compare]).all()

if are_values_equal:
    print(f"All values in {column_to_check} are equal to all values in {column_to_compare}.")
else:
    print(f"Not all values in {column_to_check} are equal to all values in {column_to_compare}.")

Not all values in cluster_exp1_seq_sampling are equal to all values in cluster_exp4_sim_sampling.


In [15]:
merged_df.to_excel("final_clusters.xlsx",index=False)